# Day 2 · 6교시 · End-to-End 통합 · 정리

> **VLM 경량화 2일 과정 · Day 2 (6교시) · 통합/정리**
> 실습 환경: **RunPod A100 80GB** · 커널: **`Python (vlm_quant)`**

---

## 이 교시의 목표
- **QLoRA → merge → AWQ → serve** 전체 파이프라인을 하나로 잇고 산출물을 점검한다.
- **4B-AWQ vs 8B-AWQ**를 정확도·크기·속도 관점에서 종합 비교한다.
- 상황별 **경량화 의사결정 가이드**를 도출한다.
- 다음 단계(GPTQ/FP8 등)를 안내하고 2일 과정을 회고한다.


## 0. 공통 헤더 — RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN 로드

> 📌 **모든 Day 2 노트북은 이 셀을 먼저 실행합니다.** RunPod 영구 볼륨의 작업 폴더 **`/workspace/VLM_FT_2`** 를 기준으로 잡고, `.env`의 **HF_TOKEN**을 불러옵니다. (Day2는 RunPod이라 Google Drive 마운트가 없습니다.)
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(모델·AWQ 결과·평가/벤치 JSON이 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다. `login()`은 호출하지 않습니다(환경변수만으로 충분).

In [5]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN(.env) 로드
#  (모든 Day2 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) RunPod 영구 볼륨의 작업 폴더 VLM_FT_2 (교시 간 모델·결과 공유). Drive 마운트 불필요.
VLM_DIR = '/workspace/VLM_FT_2'
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 모델만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /workspace/VLM_FT_2


**▸ 실행 결과 읽기.** `HF_TOKEN: 로드됨` + 작업 폴더 경로가 보이면 공통 헤더 준비 완료입니다. 이 노트북은 앞 교시들이 남긴 산출물·JSON을 모아 정리하므로, **Day2-3~2-5를 먼저 실행**해 두는 것이 전제입니다.

## 1. 전체 파이프라인 — 우리가 지나온 길

```
  [Day 1 · T4]                          [Day 2 · A100]
                                                                  
  Qwen3-VL-4B                                                     
     │ (1) 4bit 로드 (NF4)                                        
     ▼                                                            
  QLoRA 학습 ── ChartQA ──▶ LoRA 어댑터                           
     │ (2) merge_and_unload (fp16)                                
     ▼                                                            
  병합 4B ──── HF Hub(비공개) push ───────┐                       
                                          │ (3) 반입               
                                          ▼                        
                           AWQ 양자화 (llm-compressor, W4A16)     
                              · calibration: ChartQA              
                              · 비전 타워 ignore                  
                                          │ (4)                    
                                          ▼                        
                           AWQ 4bit 모델 ──▶ vLLM 서빙 ──▶ OpenAI API
```

- **학습 경량화(Day1)**: 무거운 base를 4bit로 묶고 가벼운 어댑터만 학습 → T4로 파인튜닝.
- **배포 경량화(Day2)**: 학습 결과를 AWQ 4bit로 변환 → vLLM으로 빠르게 서빙.

## 2. 산출물 점검

파이프라인이 남긴 결과물이 제자리에 있는지 확인합니다.

In [6]:
# ── 산출물 체크리스트 ───────────────────────────────────────
import os

WORKDIR = VLM_DIR   # 상단 공통 헤더의 작업 폴더 변수 사용
artifacts = {
    '4B-AWQ (로컬)': f'{WORKDIR}/Qwen3-VL-4B-ChartQA-AWQ',
    '8B-AWQ (로컬)': f'{WORKDIR}/Qwen3-VL-8B-AWQ',
}

def dir_size_gb(p):
    if not os.path.isdir(p): return None
    return sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(p) for f in fs
               if os.path.isfile(os.path.join(r, f))) / 1024**3

for name, path in artifacts.items():
    sz = dir_size_gb(path)
    status = f'✅ {sz:.2f} GB' if sz else '❌ 없음'
    print(f'{name:<16} {status}   ({path})')

print('\n· 병합 4B는 Day1-5에서 HF Hub(비공개)에 push되어 있어야 합니다.')
print('· 두 AWQ가 없으면 Day2-3을 먼저 실행하세요.')
# 저장된 결과 파일(Day2-4/2-5)도 확인
for _fn in ['eval_results.json', 'serving_bench.json']:
    _fp = f'{WORKDIR}/{_fn}'
    print(f'{_fn:<22} {"✅ 있음" if os.path.exists(_fp) else "❌ 없음"}   ({_fp})')

4B-AWQ (로컬)      ✅ 3.98 GB   (/workspace/VLM_FT_2/Qwen3-VL-4B-ChartQA-AWQ)
8B-AWQ (로컬)      ✅ 6.74 GB   (/workspace/VLM_FT_2/Qwen3-VL-8B-AWQ)

· 병합 4B는 Day1-5에서 HF Hub(비공개)에 push되어 있어야 합니다.
· 두 AWQ가 없으면 Day2-3을 먼저 실행하세요.
eval_results.json      ✅ 있음   (/workspace/VLM_FT_2/eval_results.json)
serving_bench.json     ✅ 있음   (/workspace/VLM_FT_2/serving_bench.json)


**▸ 실행 결과 읽기 — 산출물 체크리스트.**
- `4B-AWQ (로컬) ✅ 3.98 GB`, `8B-AWQ (로컬) ✅ 6.74 GB` — 두 AWQ 산출물이 제자리에 있고 크기도 Day2-3 결과와 일치합니다(`❌ 없음`이면 Day2-3을 먼저 실행).
- `eval_results.json ✅ 있음`, `serving_bench.json ✅ 있음` — Day2-4(평가)·Day2-5(서빙)의 저장본이 있어 **다음 종합표 셀이 자동으로 값을 끌어올 수 있는** 상태입니다.
- **병합 4B**는 로컬이 아니라 **HF Hub(비공개)** 에 있으므로 체크리스트에선 경로 대신 안내문만 출력됩니다.

## 3. 4B-AWQ vs 8B-AWQ 종합 비교

앞 교시(Day2-4 정확도/크기, Day2-5 서빙)에서 **측정한 값을 아래 표에 입력**하면 종합표가 만들어집니다. (아래는 예시 자리값 — 본인 측정값으로 바꾸세요.)

In [7]:
# ── 측정값 입력 후 종합표 생성 ──────────────────────────────
# Day2-4(eval_results.json)·Day2-5(serving_bench.json) 저장본을 자동 로드.
# 파일이 없으면 아래 자리값(0.0)을 본인 측정치로 직접 채우세요.
import os, json as _json
WORKDIR = VLM_DIR   # 상단 공통 헤더의 작업 폴더 변수 사용

measured = {
    '4B 원본': dict(acc=0.0, size=0.0, vram=0.0, tp=0.0),
    '4B AWQ':  dict(acc=0.0, size=0.0, vram=0.0, tp=0.0),
    '8B 원본': dict(acc=0.0, size=0.0, vram=0.0, tp=0.0),
    '8B AWQ':  dict(acc=0.0, size=0.0, vram=0.0, tp=0.0),
}

# 평가 결과(정확도·크기) 자동 반영
_evalp = f'{WORKDIR}/eval_results.json'
if os.path.exists(_evalp):
    for k, v in _json.load(open(_evalp, encoding='utf-8')).items():
        if k in measured:
            measured[k]['acc']  = v.get('relaxed_acc', 0.0)
            measured[k]['size'] = v.get('size_gb', 0.0)
    print('eval_results.json 반영됨')

# 서빙 벤치(VRAM·처리량) 자동 반영
#   Day2-5는 '8B AWQ'만 라이브 측정해 아래 구조로 저장함:
#   {'8B AWQ (serving)': {'serving_vram_gb':..., 'throughput_rps':..., 'latency_s':...}, 'offline_eval': {...}}
#   → 최상위 키 '8B AWQ (serving)' 와 내부 키 serving_vram_gb/throughput_rps 를 정확히 매핑.
_benchp = f'{WORKDIR}/serving_bench.json'
if os.path.exists(_benchp):
    _serv = _json.load(open(_benchp, encoding='utf-8'))
    _a = _serv.get('8B AWQ (serving)')
    if _a:
        measured['8B AWQ']['vram'] = _a.get('serving_vram_gb', 0.0)
        measured['8B AWQ']['tp']   = _a.get('throughput_rps', 0.0)
    print('serving_bench.json 반영됨')

print(f'{"모델":<10}{"Acc":>8}{"크기GB":>9}{"VRAM":>8}{"req/s":>8}')
print('-' * 43)
for name, m in measured.items():
    print(f'{name:<10}{m["acc"]:>8.3f}{m["size"]:>9.1f}{m["vram"]:>8.1f}{m["tp"]:>8.2f}')

eval_results.json 반영됨
serving_bench.json 반영됨
모델             Acc     크기GB    VRAM   req/s
-------------------------------------------
4B 원본        0.745      8.3     0.0    0.00
4B AWQ       0.710      4.0     0.0    0.00
8B 원본        0.000     16.3     0.0    0.00
8B AWQ       0.000      6.7    70.7    9.99


**▸ 실행 결과 읽기 — 종합표.**

표는 Day2-4(`eval_results.json`)·Day2-5(`serving_bench.json`) 저장값을 자동 반영합니다. 네 갈래로 읽습니다.

**① 4B 정확도 — 정상이자 핵심 성과.** 원본 0.745 → AWQ 0.710, **손실 3.5%p**. 크기·VRAM은 절반으로 줄면서 정확도는 거의 유지 → **AWQ의 목적(작아지되 품질 보존) 달성.**

**② 크기 — 양자화 효과.** 8.3→4.0(4B), 16.3→6.7GB(8B). 4bit인데 ~2배인 이유는 **언어 디코더 선형층만** 줄이고 **비전 타워·`lm_head`는 보존**했기 때문(Day2-3 참고).

**③ 8B 정확도 0.000 — 양자화 탓이 아님.** 같은 채점기로 4B는 멀쩡하므로 채점기는 정상입니다. 원인은 **8B가 파인튜닝 안 한 base 모델**이라 *"Based on the provided bar chart…"* 처럼 장황하게 답하고, 숫자 채점기(`relaxed_match`)가 숫자를 못 뽑아 전부 오답 처리된 것입니다(4B는 Day1에서 ChartQA로 파인튜닝됨).

**④ 서빙 열(VRAM·req/s) — 8B AWQ에만 값.** 키 매핑을 고쳐 채워진 **VRAM 70.7GB**는 vLLM이 `--gpu-memory-utilization 0.9`로 **KV 캐시까지 예약**한 서빙 점유량(모델 크기 6.7GB와 다름), **9.99 req/s**는 동시성 16 라이브 처리량입니다. 나머지 행이 0인 건 **정상** — Day2-5가 **8B AWQ만** 라이브로 띄워 측정하는 A안 설계이기 때문입니다.

---

**표를 완전히 채우려면** (지금은 의도된 공백/알려진 이슈):
- **8B 정확도** → Day2-4 평가에 *"숫자만 답하라"* system prompt + **숫자 추출 파서**를 추가하면 정상 점수가 나옵니다.
- **4B 서빙 열** → Day2-5에서 **4B AWQ도 라이브 측정**해 `serving_bench.json` 에 추가하면 채워집니다.

## 4. 경량화 의사결정 가이드

무엇을 우선하느냐에 따라 선택이 달라집니다.

| 우선순위 | 추천 | 이유 |
|---|---|---|
| **최소 메모리·최저 비용** | **4B-AWQ** | 가장 작고 빠름. 엣지·다중 인스턴스에 유리 |
| **최고 품질** | **8B 원본 / 8B-AWQ** | 8B가 차트 이해에 유리. 품질 손실 적으면 8B-AWQ |
| **품질·효율 균형** | **8B-AWQ** | 8B 품질을 4bit로 들고 가며 서빙 효율↑ |
| **학습만 가볍게** | **QLoRA(병합 4B/8B)** | 배포가 큰 GPU면 양자화 생략 가능 |

> 실무 결정 흐름: ① 품질 하한을 정한다 → ② 그 하한을 만족하는 가장 **작은/빠른** 구성을 고른다. 보통 *4B-AWQ로 시작* 해 품질이 부족하면 *8B-AWQ* 로 올립니다.

## 5. 2일 과정 회고

**Day 1 — 학습을 가볍게 (T4 · QLoRA)**
1. 경량화·양자화 기초 → 2. 4bit 로드·베이스라인 → 3. 전처리·라벨 마스킹 → 4. **QLoRA 학습** → 5. 평가·병합·Hub → 6. 8B 도전

**Day 2 — 배포를 가볍게 (A100 · AWQ + vLLM)**
1. PTQ·AWQ 이론 → 2. 환경·모델 반입 → 3. **AWQ 생성** → 4. 양자화 평가 → 5. vLLM 서빙·벤치 → 6. 통합·정리

**핵심 메시지**
> 경량화는 두 갈래입니다 — **학습(QLoRA)** 과 **배포(AWQ)**. QLoRA로 작은 GPU에서 파인튜닝하고, AWQ+vLLM으로 그 결과를 싸고 빠르게 서빙합니다. 같은 ChartQA 한 줄기로 *학습→병합→양자화→서빙* 을 끝까지 이었습니다.

수고하셨습니다. 🎉

> ✅ **최종 체크포인트**: ① 전체 파이프라인을 말로 설명할 수 있다 ② 4B-AWQ/8B-AWQ를 상황에 맞게 고를 수 있다 ③ 다음 단계(GPTQ/FP8/MoE)를 안다.